## Daily AR (Customer Outstanding) - Transformation & Consolidation

## Importing required libraries

In [1]:
import pandas as pd
import numpy as np
import glob
import datetime
import calendar
import re
import xlsxwriter
#from datetime import datetime
from spire.xls import *
from spire.xls.common import *
import os

In [2]:
current_datetime = datetime.datetime.now().strftime("%d-%b-%Y-%H%M%S")
current_datetime

'22-Sep-2026-171552'

#### Specifying Outstanding Data Source

In [3]:
source = r"D:\Power BI Data\Unapplied Movement\New\Outstanding Report\Raw"   # Daily AR

#### Extracting all files from given path

In [4]:
all_files = glob.glob(r""+source + "\\*.xlsx")

#### Listing available files from the source

In [5]:
all_files

['D:\\Power BI Data\\Unapplied Movement\\New\\Outstanding Report\\Raw\\Outstanding-Ahmedabad.xlsx',
 'D:\\Power BI Data\\Unapplied Movement\\New\\Outstanding Report\\Raw\\Outstanding-Bangalore.xlsx',
 'D:\\Power BI Data\\Unapplied Movement\\New\\Outstanding Report\\Raw\\Outstanding-Bhubaneswar.xlsx',
 'D:\\Power BI Data\\Unapplied Movement\\New\\Outstanding Report\\Raw\\Outstanding-Chandigarh.xlsx',
 'D:\\Power BI Data\\Unapplied Movement\\New\\Outstanding Report\\Raw\\Outstanding-Chennai.xlsx',
 'D:\\Power BI Data\\Unapplied Movement\\New\\Outstanding Report\\Raw\\Outstanding-Delhi.xlsx',
 'D:\\Power BI Data\\Unapplied Movement\\New\\Outstanding Report\\Raw\\Outstanding-Hyderabad.xlsx',
 'D:\\Power BI Data\\Unapplied Movement\\New\\Outstanding Report\\Raw\\Outstanding-Jaipur.xlsx',
 'D:\\Power BI Data\\Unapplied Movement\\New\\Outstanding Report\\Raw\\Outstanding-Kochi.xlsx',
 'D:\\Power BI Data\\Unapplied Movement\\New\\Outstanding Report\\Raw\\Outstanding-Kolkata.xlsx',
 'D:\\Power 

In [6]:
def consolidate_reports(all_files, header_row=0, skip_sheets=[]):
    all_df = []
    
    for file in all_files:
        file_n = pd.ExcelFile(file)
        sheet_names = file_n.sheet_names
        sheet_count = len(sheet_names)
        
        data_sheets = [i for i in range(sheet_count) if i not in skip_sheets]
        
        if not data_sheets:
            continue
        
        if len(data_sheets) > 1:
            lst_of_df = []
            headers = None
            
            for i, sheet in enumerate(data_sheets):
                df = pd.read_excel(file, sheet_name=sheet, header=header_row if i == 0 else None)
                if i == 0:
                    headers = df.columns
                else:
                    df.columns = headers[:len(df.columns)]
                lst_of_df.append(df)
            
            df = pd.concat(lst_of_df, ignore_index=True)
        else:
            df = pd.read_excel(file, sheet_name=data_sheets[0], header=header_row)
        all_df.append(df)
    
    if not all_df:
        return pd.DataFrame()
    return pd.concat(all_df, ignore_index=True)
consolidated = consolidate_reports(all_files, header_row=0, skip_sheets=[0])

In [7]:
consolidated.head()

,Business Unit,Customer Number,Customer Name,Customer Category,Bill Number,Bill Date,Type,Patient Name,Bill Amount,Settlement,Outstanding
0,Ahmedabad,5,SBI GENERAL INSURANCE COMPANY LTD,Self Pay,INV-AMD-IP-1000,03-Apr-2026,AHM IP Invoice,Shaheen Afridi,12875,0,12875
1,Ahmedabad,6,MD INDIA HEALTH CARE SERVICES PSU LTD (TPA),Self Pay,INV-AMD-OP-2001,12-Mar-2026,AHM OP Invoice,Jonny Bairstow,86782,64829,21953
2,Ahmedabad,8,BAJAJ ALLIANZ GENERAL INSURANCE,TPA/Insurance,INV-AMD-IP-1001,19-Mar-2026,AHM IP Invoice,Glenn Maxwell,49569,0,49569
3,Ahmedabad,4,ICICI LOMBARD GENERAL INSURANCE,Corporate,INV-AMD-OP-2004,12-Apr-2026,AHM OP Invoice,Lalo Salamanca,78136,56348,21788
4,Ahmedabad,1,MEDI ASSIST-GO DIGIT GENERAL INSURANCE LTD,Corporate,INV-AMD-OP-2014,07-Feb-2026,AHM OP Invoice,Mike Ehrmantraut,31812,0,31812


## Month Year mapping
#### Extracts month and year from date column

In [8]:
consolidated["Bill Date"] = pd.to_datetime(consolidated["Bill Date"], format="%d-%b-%Y")

consolidated["Month_Year"] = consolidated["Bill Date"].dt.strftime("%b-%y")

In [9]:
consolidated.head()

,Business Unit,Customer Number,Customer Name,Customer Category,Bill Number,Bill Date,Type,Patient Name,Bill Amount,Settlement,Outstanding,Month_Year
0,Ahmedabad,5,SBI GENERAL INSURANCE COMPANY LTD,Self Pay,INV-AMD-IP-1000,2026-04-03,AHM IP Invoice,Shaheen Afridi,12875,0,12875,Apr-26
1,Ahmedabad,6,MD INDIA HEALTH CARE SERVICES PSU LTD (TPA),Self Pay,INV-AMD-OP-2001,2026-03-12,AHM OP Invoice,Jonny Bairstow,86782,64829,21953,Mar-26
2,Ahmedabad,8,BAJAJ ALLIANZ GENERAL INSURANCE,TPA/Insurance,INV-AMD-IP-1001,2026-03-19,AHM IP Invoice,Glenn Maxwell,49569,0,49569,Mar-26
3,Ahmedabad,4,ICICI LOMBARD GENERAL INSURANCE,Corporate,INV-AMD-OP-2004,2026-04-12,AHM OP Invoice,Lalo Salamanca,78136,56348,21788,Apr-26
4,Ahmedabad,1,MEDI ASSIST-GO DIGIT GENERAL INSURANCE LTD,Corporate,INV-AMD-OP-2014,2026-02-07,AHM OP Invoice,Mike Ehrmantraut,31812,0,31812,Feb-26


## Ageing Bucket

### Getting Current Date

In [10]:
current_date = pd.to_datetime(datetime.datetime.now().date(), format='%d:%b:%y:%H:%M:%S')
current_date

Timestamp('2026-09-22 00:00:00')

### Calculating date difference between bill date and current date

##### If Daily AR : current_date <br> If Monthly AR : consolidated["Bill Date"].max()

In [11]:
consolidated["Ageing"] = ( current_date - consolidated["Bill Date"] ).dt.days

#### Grouping the date difference into a Ageing Bucket

In [12]:
def Ageing_Bucket(consolidated):
    if consolidated["Ageing"] <= 30:
        return "1) 0D-30D"
    elif consolidated["Ageing"] <= 60:
        return "2) 31D-60D"
    elif consolidated["Ageing"] <= 90:
        return "3) 61D-90D"
    elif consolidated["Ageing"] <= 120:
        return "4) 91D-120D"
    elif consolidated["Ageing"] <= 180:
        return "5) 121D-180D"
    elif consolidated["Ageing"] <= 365:
        return "6) 181D-1Y"
    elif consolidated["Ageing"] <= 730:
        return "7) 1Y-2Y"
    elif consolidated["Ageing"] <= 1095:
        return "8) 2Y-3Y"
    elif consolidated["Ageing"] > 1095:
        return "9) 3Y+"
    else:
        return "N/A"

In [13]:
consolidated["Ageing Bucket"] = consolidated.apply(Ageing_Bucket,axis=1)

#### Overview of Consolidated DataFrame after the transformations

In [14]:
consolidated.head(1)

,Business Unit,Customer Number,Customer Name,Customer Category,Bill Number,Bill Date,Type,Patient Name,Bill Amount,Settlement,Outstanding,Month_Year,Ageing,Ageing Bucket
0,Ahmedabad,5,SBI GENERAL INSURANCE COMPANY LTD,Self Pay,INV-AMD-IP-1000,2026-04-03,AHM IP Invoice,Shaheen Afridi,12875,0,12875,Apr-26,172,5) 121D-180D


## Mapping Std Name and Std Category to Consolidated from Payer Master 

In [15]:
Payer_Master = pd.read_excel(r"D:\Power BI Data\Unapplied Movement\New\Payer_Customer_Master.xlsx")

In [16]:
consolidated = consolidated.merge(Payer_Master , left_on = "Customer Name",right_on = "Payer_Name_Raw", how = "left")

In [17]:
consolidated.head(2)

,Business Unit,Customer Number,Customer Name,Customer Category,Bill Number,Bill Date,Type,Patient Name,Bill Amount,Settlement,Outstanding,Month_Year,Ageing,Ageing Bucket,Payer_Code,Payer_Name_Raw,Std_Payer_Name,Std_Payer_Category
0,Ahmedabad,5,SBI GENERAL INSURANCE COMPANY LTD,Self Pay,INV-AMD-IP-1000,2026-04-03,AHM IP Invoice,Shaheen Afridi,12875,0,12875,Apr-26,172,5) 121D-180D,5,SBI GENERAL INSURANCE COMPANY LTD,SBI General Insurance,TPA/Insurance
1,Ahmedabad,6,MD INDIA HEALTH CARE SERVICES PSU LTD (TPA),Self Pay,INV-AMD-OP-2001,2026-03-12,AHM OP Invoice,Jonny Bairstow,86782,64829,21953,Mar-26,194,6) 181D-1Y,6,MD INDIA HEALTH CARE SERVICES PSU LTD (TPA),MD India Healthcare Services (TPA) Pvt Ltd,TPA/Insurance


In [18]:
Blanks_Check = consolidated[consolidated["Std_Payer_Category"].isna()]
Blanks_Check.head(1)

,Business Unit,Customer Number,Customer Name,Customer Category,Bill Number,Bill Date,Type,Patient Name,Bill Amount,Settlement,Outstanding,Month_Year,Ageing,Ageing Bucket,Payer_Code,Payer_Name_Raw,Std_Payer_Name,Std_Payer_Category


#### Previous Year MM Receipt

In [19]:
Self_Paid = pd.read_excel(r"D:\Power BI Data\Unapplied Movement\New\Settlement\Accounting_Report_Dummy.xlsx")

In [20]:
Self_Paid.head(1)

,Business Unit,Payment Date,Payer Code,Payer Name,Receipt Number,Receipt Amount,Bill Cust Code,Bill Customer Name,Accounting Bill Number,Bill Date,Bill Amount,Accounting Date,Accounted Amount
0,Bhubaneswar,11-Apr-2026,1,MEDI ASSIST-GO DIGIT GENERAL INSURANCE LTD,CMS28246873582123456789,18340,1,MEDI ASSIST-GO DIGIT GENERAL INSURANCE LTD,INV-BBS-IP-1000,12-Mar-2026,63599,2026-05-01,18340


In [21]:
Self_Paid = Self_Paid[Self_Paid["Payer Name"].str.lower().str.contains("self pay")]

In [22]:
Self_Paid.head(1)

,Business Unit,Payment Date,Payer Code,Payer Name,Receipt Number,Receipt Amount,Bill Cust Code,Bill Customer Name,Accounting Bill Number,Bill Date,Bill Amount,Accounting Date,Accounted Amount
1,Kochi,25-May-2026,2,SELF PAY,NEFT66691283914123456789,30846,2,SELF PAY,INV-COK-OP-2000,13-Mar-2026,56985,2026-05-28,30846


### Aggregating MM Receipt Adjusted Value

In [23]:
Self_Paid_Agg = Self_Paid.groupby("Accounting Bill Number").agg(Self_Paid_Amount=("Accounted Amount","sum"))

In [24]:
Self_Paid_Agg.head(1)

,Self_Paid_Amount
Accounting Bill Number,
INV-AMD-OP-2007,72861


### Merging MM Applied Value in Consolidation OS DataFrame

In [25]:
consolidated = consolidated.merge(Self_Paid_Agg,
                                  left_on = "Bill Number",
                                  right_on = "Accounting Bill Number",
                                  how = "left")

In [26]:
consolidated.head(1)

,Business Unit,Customer Number,Customer Name,Customer Category,Bill Number,Bill Date,Type,Patient Name,Bill Amount,Settlement,Outstanding,Month_Year,Ageing,Ageing Bucket,Payer_Code,Payer_Name_Raw,Std_Payer_Name,Std_Payer_Category,Self_Paid_Amount
0,Ahmedabad,5,SBI GENERAL INSURANCE COMPANY LTD,Self Pay,INV-AMD-IP-1000,2026-04-03,AHM IP Invoice,Shaheen Afridi,12875,0,12875,Apr-26,172,5) 121D-180D,5,SBI GENERAL INSURANCE COMPANY LTD,SBI General Insurance,TPA/Insurance,NaN


### Calculating difference between "Bill Amount" and "MM Adjusted Amount"

In [27]:
consolidated["Self_Paid_Amount vs Bill_Amount"] = consolidated["Bill Amount"] - consolidated["Self_Paid_Amount"]

### Calculating difference between the calculated "DIFF" and "Outstanding"

In [28]:
consolidated["OS_vs_diff"] = consolidated["Self_Paid_Amount vs Bill_Amount"] - consolidated["Outstanding"]

#### Partial Paid values are not entirely Partial Paid.  MM Receipt Adjusted cases should be considered as Fully Unpaid. Not Partially Paid. That's why we are comparing MM Adjusted Value and changing the Invoice Status accordingly here!! 

In [29]:
def partial_tags(consolidated):
    if consolidated["Type"] == "Payments":
        return "Unapplied"
    elif consolidated["Outstanding"] < 0:
        return "Non-AR"
    elif consolidated["OS_vs_diff"] == 0:  # Check this FIRST before Partially Paid
        return "Fully Unpaid"
    elif (consolidated["Outstanding"] / consolidated["Bill Amount"]) < 0.995:
        return "Partially Paid"
    elif (consolidated["Outstanding"] / consolidated["Bill Amount"]) >= 0.995:
        return "Fully Unpaid"
    else:
        return "Other"

In [30]:
consolidated["Invoice Status"] = consolidated.apply(partial_tags,axis=1)

In [31]:
consolidated.head(1)

,Business Unit,Customer Number,Customer Name,Customer Category,Bill Number,Bill Date,Type,Patient Name,Bill Amount,Settlement,...,Ageing,Ageing Bucket,Payer_Code,Payer_Name_Raw,Std_Payer_Name,Std_Payer_Category,Self_Paid_Amount,Self_Paid_Amount vs Bill_Amount,OS_vs_diff,Invoice Status
0,Ahmedabad,5,SBI GENERAL INSURANCE COMPANY LTD,Self Pay,INV-AMD-IP-1000,2026-04-03,AHM IP Invoice,Shaheen Afridi,12875,0,...,172,5) 121D-180D,5,SBI GENERAL INSURANCE COMPANY LTD,SBI General Insurance,TPA/Insurance,NaN,NaN,NaN,Fully Unpaid


### Claim ID Mapping

##### Claim ID Data from CB Team

In [32]:
Claim_ID = pd.read_excel(r"D:\Power BI Data\Unapplied Movement\New\Claim Submission\Claim_Submission_Report_Dummy.xlsx")

In [33]:
Claim_ID.head(1)

,Business Unit,Claim ID,Claim Submission Date,Claim Payer Name,Claim Amount,Claim Bill Number,Bill Date,Bill Amount,Bill Payer Name,Patient Name,UTR Number
0,Bhubaneswar,CLM-BBS-001001,13-Mar-2026,MEDI ASSIST-GO DIGIT GENERAL INSURANCE LTD,18340,INV-BBS-IP-1000,12-Mar-2026,63599,MEDI ASSIST-GO DIGIT GENERAL INSURANCE LTD,Jimmy McGill,CMS28246873582123456789


In [34]:
consolidated = consolidated.merge(Claim_ID[["Claim Bill Number" , "Claim ID"]] , left_on="Bill Number" , right_on="Claim Bill Number")

In [43]:
consolidated.head(1)

,Business Unit,Customer Number,Customer Name,Customer Category,Bill Number,Bill Date,Type,Patient Name,Bill Amount,Settlement,...,Ageing Bucket,Payer_Code,Std_Payer_Name,Std_Payer_Category,Self_Paid_Amount,Self_Paid_Amount vs Bill_Amount,OS_vs_diff,Invoice Status,Claim ID,Financial Year
0,Ahmedabad,8,BAJAJ ALLIANZ GENERAL INSURANCE,TPA/Insurance,INV-AMD-IP-1001,2026-03-19,AHM IP Invoice,Glenn Maxwell,49569,0,...,6) 181D-1Y,8,Bajaj Allianz General Insurance Company Limited,TPA/Insurance,NaN,NaN,NaN,Fully Unpaid,CLM-AMD-001002,FY 25-26


In [36]:
consolidated = consolidated.drop(columns={"Payer_Name_Raw" , "Claim Bill Number"})

### Converting payment values into numeric values

In [44]:
def convert_to_number(row):
    if row["Type"]=="Payments":
        return pd.to_numeric(row["Bill Number"], errors='coerce')
    else:
        return row["Bill Number"]

In [45]:
consolidated["Bill Number"] = consolidated.apply(convert_to_number, axis='columns').fillna(consolidated["Bill Number"])

### Financial Year Mapping

In [46]:
def FY_assign(consolidated):
    month = consolidated["Bill Date"].month
    year = consolidated["Bill Date"].year
    
    if consolidated["Bill Date"] < pd.Timestamp("2021-04-01"):
        return "Prior"
    
    elif month >= 4:
        return f"FY {str(year)[-2:]}-{str(year+1)[-2:]}"
    
    else:
        return f"FY {str(year-1)[-2:]}-{str(year)[-2:]}"

In [47]:
consolidated["Financial Year"] = consolidated.apply(FY_assign,axis=1)

In [48]:
consolidated.head(1)

,Business Unit,Customer Number,Customer Name,Customer Category,Bill Number,Bill Date,Type,Patient Name,Bill Amount,Settlement,...,Ageing Bucket,Payer_Code,Std_Payer_Name,Std_Payer_Category,Self_Paid_Amount,Self_Paid_Amount vs Bill_Amount,OS_vs_diff,Invoice Status,Claim ID,Financial Year
0,Ahmedabad,8,BAJAJ ALLIANZ GENERAL INSURANCE,TPA/Insurance,INV-AMD-IP-1001,2026-03-19,AHM IP Invoice,Glenn Maxwell,49569,0,...,6) 181D-1Y,8,Bajaj Allianz General Insurance Company Limited,TPA/Insurance,NaN,NaN,NaN,Fully Unpaid,CLM-AMD-001002,FY 25-26


### Writing the DataFrame into an Excel file

In [49]:
with pd.ExcelWriter(r"D:\Power BI Data\Unapplied Movement\New\Claim Submission\Consolidated\Consolidated Outstanding-"+current_datetime+".xlsx", engine='xlsxwriter', datetime_format='DD-MMM-YY') as writer:
   consolidated.to_excel(writer, sheet_name="Consolidated", index=False)

In [ ]:
################################################################# THE END ###############################################################################